# 🎯 Objetivos

Filtrado y limpieza inicial de los datos para obtener las tablas sin casos que no entran en el estandar de calidad predefinido.

* ⚙️ **Configuración**. Rutas, paquetes y funciones.
* 🗃️ **Revisión inicial de las tablas**. Filas por cada tabla.
* ♻️ **Carga de datos**.
* 🧪 **Filtrado de casos**. Aplicación de umbrales específicos en las tablas games y user_games.
* 💾 **Guardado de información**. Guardado de información y resumen de los datos almacenados.

# ⚙️ Configuración inicial

## Rutas (paths)

In [1]:
import os

# Rutas en función de la carpeta raíz del proyecto (README.md)
base_path = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in locals() else os.path.abspath("..")

# Subrutas
data_dir = os.path.join(base_path, "data")
db_path = os.path.join(data_dir, "steam.db")
src_dir = os.path.join(base_path, "src")

## Paquetes y funciones

In [2]:
import sqlite3
import pandas as pd

# 🗃️ Revisión inicial de las tablas

In [3]:
# Conexión
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Tablas
tablas = ("games", "user_games", "user_reviews", "users")

In [4]:
# FILAS POR TABLA
query = f"""
SELECT name FROM sqlite_master
WHERE type='table' AND name IN {tablas};
"""
cur.execute(query)
tables = [row[0] for row in cur.fetchall()]

# Mostrar número de filas
for t in tables:
    cur.execute(f"SELECT COUNT(*) FROM {t};")
    count = cur.fetchone()[0]
    print(f"{t}: {count} filas")

games: 54294 filas
users: 226757 filas
user_reviews: 296133 filas
user_games: 11808514 filas


# ♻️ Carga de datos

In [5]:
# Cargamos los datos de SQLite

games = pd.read_sql_query("SELECT * FROM games;", conn)
users = pd.read_sql_query("SELECT * FROM users;", conn)
user_games = pd.read_sql_query("SELECT * FROM user_games;", conn)

In [6]:
# Formatos a int
user_games["user_id"] = user_games["user_id"].astype(int)
user_games["appid"]   = user_games["appid"].astype(int)
games["appid"]        = games["appid"].astype(int)

# 🧪 Filtrado de casos

## Tabla games
Filtramos la tabla games. Nos vamos a quedar con aquellos juegos que:

* Tengan 1000 reseñas o más.
* Al menos 50 usuarios únicos.

In [7]:
# Juegos con más de 1000 reseñas o más
games_filtered = games[games["total_reviews"] >= 1000]

# Usuarios por juego
users_per_game = (
    user_games[user_games["appid"].isin(games_filtered["appid"])]
    .groupby("appid")["user_id"]
    .nunique()
    .reset_index(name="num_users")
    .sort_values(by="num_users", ascending=False)
)

# Juegos con al menos 50 usuarios únicos
valid_games = users_per_game[users_per_game["num_users"] >= 50]["appid"]
games_final = games_filtered[games_filtered["appid"].isin(valid_games)]

## Tabla user_game
Filtramos la tabla user_game. Nos vamos a quedar con aquellos usuarios que:

* Tengan alguno de los juegos de la tabla de games filtrada.
* Hayan jugado al menos a 5 juegos de la tabla de games filtrada.

In [8]:
# Relaciones usuario-juego solo de los juegos válidos
user_games_filtered = user_games[user_games["appid"].isin(games_final["appid"])]

# Número de juegos válidos de cada usuario
games_per_user = (
    user_games_filtered.groupby("user_id")["appid"]
    .nunique()
    .reset_index(name="num_games")
)

# Usuarios con 5 o más de esos juegos
active_users = games_per_user[games_per_user["num_games"] >= 5]["user_id"]
user_games_final = user_games_filtered[user_games_filtered["user_id"].isin(active_users)]

# Chequeo
print(user_games_final.groupby("user_id")["appid"]
    .nunique()
    .reset_index(name="num_games")
    .sort_values(by="num_games", ascending=False) )

                 user_id  num_games
35434  76561198367471798       2388
29460  76561198237402290       2123
10766  76561198035865245       2106
10106  76561198031164839       2101
622    76561197966082557       2050
...                  ...        ...
48378  76561199856467970          5
48440  76561199873504573          5
46002  76561199375656115          5
45999  76561199375555793          5
48453  76561199881985199          5

[48473 rows x 2 columns]


# 💾 Guardado de información

In [9]:
# Guardamos las versiones finales
games_final.to_sql("games_filtered", conn, if_exists="replace", index=False)

active_users_df = pd.DataFrame({"user_id": active_users.astype(int)})
active_users_df.to_sql("users_filtered", conn, if_exists="replace", index=False)

user_games_final.to_sql("user_games_filtered", conn, if_exists="replace", index=False)

# Cerramos la conexión
conn.close()

## Resumen de los datos

In [10]:
print("Juegos seleccionados:", len(games_final))
print("Usuarios seleccionados:", active_users.nunique())
print("Relaciones finales:", len(user_games_final))

Juegos seleccionados: 2576
Usuarios seleccionados: 48473
Relaciones finales: 7183044
